In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

**Pre-Processing and Feature Engineering**

In [8]:
def load_and_sanitize_data(filepath):
    """Loads the dataset and handles basic structural mismatches."""
    df = pd.read_csv(filepath)
    
    # Structural Mismatches
    df = df.dropna(subset=['match_date', 'result']).copy()
    df['last_formation'] = df['last_formation'].fillna(df['starting_formation'])
    
    # Chronological Sorting (Absolute prerequisite for time-series)
    df['match_date'] = pd.to_datetime(df['match_date'])
    df = df.sort_values(by=['team', 'match_date']).reset_index(drop=True)
    
    # Drop exact duplicate rows if they exist
    df = df.drop_duplicates(subset=['match_id', 'team'])
    
    return df

def handle_outliers(df):
    """Caps extreme values (Top 1% and Bottom 1%) to prevent freak matches or data errors from ruining rolling averages."""
    df_clean = df.copy()
    
    # Hard Caps for strict football realities
    df_clean['xg'] = df_clean['xg'].clip(upper=5.0)
    # A match with 3+ red cards for one team is an extreme anomaly that breaks tactical shapes
    df_clean['red_cards'] = df_clean['red_cards'].clip(upper=2) 
    
    # Two-Sided Winsorization for all volume metrics
    volume_metrics = [
        'passes', 'pressures', 'possession_events', 'shots', 
        'fouls_committed', 'ball_recoveries', 'duels', 'dribbles', 
        'interceptions', 'clearances', 'tackles'
    ]
    
    for col in volume_metrics:
        # Calculate the 99th percentile (Ceiling) and 1st percentile (Floor)
        p99 = df_clean[col].quantile(0.99)
        p01 = df_clean[col].quantile(0.01)
        
        # Clip the data so anything above p99 becomes p99, and anything below p01 becomes p01
        df_clean[col] = df_clean[col].clip(lower=p01, upper=p99)
        
    return df_clean

def engineer_match_ratios(df):
    """Engineers tactical metastats for the current match, preventing Division by Zero."""
    
    # We use a small epsilon (1e-5) to mathematically guarantee no Inf/NaN from division by zero
    epsilon = 1e-5
    
    # Pressing Efficiency (Capped at 1.0, as you cannot recover more balls than you press)
    df['pressing_efficiency'] = (df['ball_recoveries'] / (df['pressures'] + epsilon)).clip(upper=1.0)
    
    # Shot Lethality (xG per shot. Capped at 1.0)
    df['shot_quality'] = (df['xg'] / (df['shots'] + epsilon)).clip(upper=1.0)
    
    # Directness Index (Passes per possession event)
    df['directness_index'] = df['passes'] / (df['possession_events'] + epsilon)
    
    # Chaos/Discipline Index
    df['chaos_index'] = df['fouls_committed'] + (df['yellow_cards'] * 3) + (df['red_cards'] * 5)
    
    return df

def calculate_prematch_profiles(df):
    """Calculates the 5-game rolling averages and contextual features."""
    
    tactical_cols = [
        'passes', 'shots', 'xg', 'pressures', 'fouls_committed', 
        'ball_recoveries', 'interceptions', 'clearances', 'tackles', 
        'possession_events', 'pressing_efficiency', 'shot_quality', 
        'directness_index', 'chaos_index'
    ]
    
    def apply_rolling_logic(group):
        # Rolling Mean (Shifted)
        rolled_mean = group[tactical_cols].rolling(window=5, min_periods=1).mean().shift(1)
        
        # Rolling Volatility (Standard Deviation of key stats to measure consistency)
        rolled_std = group[['xg', 'pressures']].rolling(window=5, min_periods=2).std().shift(1)
        rolled_std = rolled_std.rename(columns={'xg': 'xg_volatility', 'pressures': 'pressures_volatility'})
        
        # Momentum (3-game form vs 10-game baseline)
        short_xg = group['xg'].rolling(window=3, min_periods=1).mean().shift(1)
        long_xg = group['xg'].rolling(window=10, min_periods=1).mean().shift(1)
        momentum = short_xg - long_xg
        
        # Combine all engineered time-series features
        result = pd.concat([rolled_mean, rolled_std], axis=1)
        result['xg_momentum'] = momentum
        return result

    # Apply the rolling logic grouped by team
    rolling_features = df.groupby('team', group_keys=False).apply(apply_rolling_logic)
    
    # Rename the mean columns to indicate they are rolling averages
    rename_dict = {col: f"{col}_rolling" for col in tactical_cols}
    rolling_features = rolling_features.rename(columns=rename_dict)
    
    # Merge the rolling features back onto the main dataframe
    df_final = pd.concat([df, rolling_features], axis=1)
    
    # Calculate Contextual Features (Location & Rest)
    df_final['is_home'] = np.where(df_final['home_away'] == 'home', 1, 0)
    
    # Calculate days rest, filling the first game with 7 days, and capping extreme gaps (summer breaks) at 14 days
    df_final['days_rest'] = df_final.groupby('team')['match_date'].diff().dt.days.fillna(7).clip(upper=14)
    
    return df_final

def finalize_dataset(df):
    """Cleans up the final dataframe for ML ingestion."""
    # Drop rows where rolling stats are NaN (i.e., the very first match for every team)
    df_final = df.dropna(subset=['xg_rolling']).copy()
    
    # For matches 2-4, standard deviation (volatility) might be NaN, fill with 0
    df_final['xg_volatility'] = df_final['xg_volatility'].fillna(0)
    df_final['pressures_volatility'] = df_final['pressures_volatility'].fillna(0)
    df_final['xg_momentum'] = df_final['xg_momentum'].fillna(0)
    
    print(f"Final dataset shape: {df_final.shape}")
    return df_final


if __name__ == "__main__":
    # Run the pipeline
    raw_df = load_and_sanitize_data('Merged_Aggregate.csv')
    sane_df = handle_outliers(raw_df)
    ratio_df = engineer_match_ratios(sane_df)
    profile_df = calculate_prematch_profiles(ratio_df)
    final_prematch_df = finalize_dataset(profile_df)
    
    # Save the master dataset
    final_prematch_df.to_csv('Engineered_PreMatch_Data.csv', index=False)

Final dataset shape: (1411, 52)
